# Photon reconstruction

In [1]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DOCS = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(DOCS / 'scripts'))
import reconstruction
importlib.reload(reconstruction)
from reconstruction import (
    load_visibility,
    load_reconstruction_frame,
    load_reconstruction_frames,
    derive_pmt_calibration,
    reconstruct_event,
    track_row,
)
from selection import select

#Update these paths to your local ROOT files before running the full reconstruction
LIBRARY = DOCS.parent / 'data' / 'LibraryData' / 'uboone_photon_library_v6_70kV.root'
VIS_CACHE = DOCS / 'cache' / 'vis_lookup.pkl'
DATA_FILES = [
    DOCS.parent / 'data' / 'on beam' / 'run4b_bnb_beam_on_pandora_v73_reco2_final_reco2_reco2_beam_good',
]
MC_FILE = DOCS.parent / 'data' / 'simulated' / 'run4b_bnb_nu_overlay_unified_pandora_reco2_run4b_reco2'
PE_CUT = 8
N_BINS = 30

In [ ]:
#Building the library cache is very very slow, so want to try to save it and reuse if possible
visibility = None
if LIBRARY.exists():
    VIS_CACHE.parent.mkdir(exist_ok=True)
    visibility = load_visibility(LIBRARY, VIS_CACHE)
    print(f'Loaded {len(visibility)} library voxels.')
else:
    print(f'Photon library not found: {LIBRARY}')
    print('Update LIBRARY, then rerun this cell.')

Photon library not found: c:\Users\rmcot\Documents\latelight-main\latelight-main\data\LibraryData\uboone_photon_library_v6_70kV.root
Update LIBRARY, then rerun this cell.


In [3]:
# Load the event files and apply the same reconstructed selection to data and MC
data_paths = [path for path in DATA_FILES if path.with_suffix('.root').exists() or path.exists()]
if data_paths:
    data_frame = load_reconstruction_frames(data_paths)
    selected_protons = select(data_frame, 'proton')
    selected_muons = select(data_frame, 'muon')
    print(f'Data: {len(selected_protons)} proton and {len(selected_muons)} muon candidates.')
else:
    selected_protons = None
    selected_muons = None
    print('No data ROOT files found. Update DATA_FILES, then rerun this cell.')

if MC_FILE.with_suffix('.root').exists() or MC_FILE.exists():
    mc_frame = load_reconstruction_frame(MC_FILE)
    mc_protons = select(mc_frame, 'proton')
    mc_muons = select(mc_frame, 'muon')
    print(f'MC: {len(mc_protons)} proton and {len(mc_muons)} muon candidates.')
else:
    mc_protons = None
    mc_muons = None
    print('No MC ROOT file found. Update MC_FILE, then rerun this cell.')

No data ROOT files found. Update DATA_FILES, then rerun this cell.
MC: 1668 proton and 1365 muon candidates.


In [4]:
#relative PMT efficiencies from the selected 400-600 MeV muons
calibration = None
if visibility is not None and selected_muons is not None:
    calibration = derive_pmt_calibration(
        selected_muons,
        visibility,
        pe_cut=PE_CUT,
        n_bins=N_BINS,
        ke_low_mev=400,
        ke_high_mev=600,
    )
    print(f'Calibrated {np.isfinite(calibration).sum()} of 32 PMTs.')
else:
    print('Load the visibility library and selected muons before deriving calibration.')

Load the visibility library and selected muons before deriving calibration.


## Reconstruction formula

For PMT $n$, the event estimate is $N_n = PE_n / (V_n f_n)$, where $V_n$ is the emission-weighted visibility and $f_n$ is the relative PMT calibration. PMTs below the PE threshold or with invalid visibility are removed before the 10-90% trimmed mean is calculated.

In [5]:
def reconstruct_dataframe(frame, particle, calibration, visibility):
    rows = []
    if frame is None or calibration is None or visibility is None:
        return pd.DataFrame(columns=['kinetic_energy_mev', 'photons', 'photons_per_mev', 'n_pmts_used'])
    for _, row in frame.iterrows():
        start, end, pe, ke_mev = track_row(row, particle)
        photons, n_pmts = reconstruct_event(
            start, end, pe, calibration, visibility, ke_mev, particle, PE_CUT, N_BINS
        )
        rows.append({
            'kinetic_energy_mev': ke_mev,
            'photons': photons,
            'photons_per_mev': photons / ke_mev if np.isfinite(photons) and ke_mev > 0 else np.nan,
            'n_pmts_used': n_pmts,
        })
    return pd.DataFrame(rows)

# Reconstruct data and MC
data_protons = reconstruct_dataframe(selected_protons, 'proton', calibration, visibility)
data_muons = reconstruct_dataframe(selected_muons, 'muon', calibration, visibility)
mc_protons_reco = reconstruct_dataframe(mc_protons, 'proton', calibration, visibility)
mc_muons_reco = reconstruct_dataframe(mc_muons, 'muon', calibration, visibility)
print(f'Data: {len(data_protons)} protons, {len(data_muons)} muons')
print(f'MC: {len(mc_protons_reco)} protons, {len(mc_muons_reco)} muons')

Data: 0 protons, 0 muons
MC: 0 protons, 0 muons


## Yield plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), sharey=True)

for ax, proton, muon, title in [
    (axes[0], data_protons, data_muons, 'Data'),
    (axes[1], mc_protons_reco, mc_muons_reco, 'MC'),
]:
    ax.scatter(proton['kinetic_energy_mev'], proton['photons_per_mev'],
               s=12, alpha=0.45, label='Protons')
    ax.scatter(muon['kinetic_energy_mev'], muon['photons_per_mev'],
               s=12, alpha=0.45, label='Muons')
    ax.set(xlabel='Kinetic energy (MeV)', ylabel='Photons / MeV', title=title)
    ax.set_xlim(50, 500)
    ax.set_ylim(0, 5000)
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
ke_bins = [(100, 150), (150, 200), (200, 250), (250, 400)]
fig, axes = plt.subplots(2, 4, figsize=(16, 8), sharey='row')

for column, (low, high) in enumerate(ke_bins):
    for row, (proton, muon, title) in enumerate([
        (data_protons, data_muons, 'Data'),
        (mc_protons_reco, mc_muons_reco, 'MC'),
    ]):
        ax = axes[row, column]
        proton_values = proton.loc[proton['kinetic_energy_mev'].between(low, high, inclusive='left'), 'photons_per_mev'].dropna()
        muon_values = muon.loc[muon['kinetic_energy_mev'].between(low, high, inclusive='left'), 'photons_per_mev'].dropna()
        combined = pd.concat([proton_values, muon_values])
        bins = np.linspace(0, combined.quantile(0.98) * 1.2, 25) if len(combined) else 20
        ax.hist(proton_values, bins=bins, alpha=0.65, label='Protons')
        ax.hist(muon_values, bins=bins, alpha=0.65, label='Muons')
        ax.set_title(f'{title}: {low}-{high} MeV')
        if column == 0:
            ax.set_ylabel('Events')
        if row == 1:
            ax.set_xlabel('Photons / MeV')
        if row == 0 and column == 0:
            ax.legend()

plt.tight_layout()
plt.show()